In [1]:
from sktime.datasets import load_from_tsfile_to_dataframe
import pandas as pd
import pywt
import numpy as np
import sklearn
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import os
import json

In [2]:
sklearn.__version__

'1.7.0'

In [3]:
def expand_and_label(x_df, y_array):
    expanded_df = pd.DataFrame(x_df['dim_0'].tolist())
    expanded_df.columns = [f'week{str(i).zfill(2)}' for i in range(expanded_df.shape[1])]
    
    expanded_df['label'] = y_array
    
    return expanded_df

def svm_train_and_test(data_name):
    
    data_id = data_name[4:6]
    file_path_train = f"data/{data_name}/TokenItaly_vers0/TokenItaly_vers0_TRAIN.ts"
    file_path_test = f"data/{data_name}/TokenItaly_vers0/TokenItaly_vers0_TEST.ts"

    x_train, y_train = load_from_tsfile_to_dataframe(file_path_train)
    x_test, y_test = load_from_tsfile_to_dataframe(file_path_test)

    train_expanded = expand_and_label(x_train, y_train)
    test_expanded = expand_and_label(x_test, y_test)
    
    # separate each data into x&y and train&test
    X_train = train_expanded.drop(columns=['label'])
    y_train = train_expanded['label']
    
    X_test = test_expanded.drop(columns=['label'])
    y_test = test_expanded['label']
    
    # make them integer
    y_train = y_train.astype(int)
    y_test = y_test.astype(int)

    # scale the data mean=0 & variance=unit
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # train SVM
    svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
    svm_model.fit(X_train_scaled, y_train)

    # evaluation
    y_pred = svm_model.predict(X_test_scaled)
    y_proba = svm_model.predict_proba(X_test_scaled)[:, 1]

    # metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_test, y_proba)
    except ValueError:
        auc = float('nan')
    
    result_dict = {
        "data_id" : data_id,
        "data_name" : data_name,
        "model" : "Support Vector Machine",
        "Accuracy" : accuracy,
        "Precision" : precision,
        "Recall" : recall,
        "F1-score" : f1,
        "ROC-AUC score" : auc
                }

    return result_dict

In [4]:
json_path = 'results.json'
csv_path = 'results.csv'

if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        all_results = json.load(f)
else:
    all_results = []

data_root_path = 'data'
data_names = sorted([
    name for name in os.listdir(data_root_path)
    if os.path.isdir(os.path.join(data_root_path, name)) and name.startswith('data')
])

c = 0
for data_name in data_names:
    result_dict = svm_train_and_test(data_name)
    all_results.append(result_dict)
    c += 1
    print(data_name)

with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2)


df = pd.DataFrame(all_results)
df.to_csv(csv_path, index=False)

print(f"{c} data is trained and results added to the files...")

data10_IC1_FM_label_wl_20_pw_0_unwo_dropped_1_1_ratio_full_data
data11_IC1_FM_label_wl_20_pw_0_unwo_dropped_1_4_ratio_full_data
data12_IC1_FM_label_wl_20_pw_0_unwo_dropped_1_1_ratio_max_5000
data13_IC1_FM_label_wl_20_pw_0_unwo_dropped_1_1_ratio_max_2500
data20_IC2_FM_label_wl_20_pw_0_unwo_dropped_1_1_ratio_full_data
data21_IC2_FM_label_wl_20_pw_0_unwo_dropped_1_4_ratio_full_data
data22_IC2_FM_label_wl_20_pw_0_unwo_dropped_1_1_ratio_max_5000
data23_IC2_FM_label_wl_20_pw_0_unwo_dropped_1_1_ratio_max_2500
data30_CJ_FM_label_wl_20_pw_0_unwo_dropped_1_1_ratio_full_data
data31_CJ_FM_label_wl_20_pw_0_unwo_dropped_1_4_ratio_full_data
data40_LGSR_FM_label_wl_20_pw_0_unwo_dropped_1_1_ratio_full_data
data41_LGSR_FM_label_wl_20_pw_0_unwo_dropped_1_4_ratio_full_data
12 data is trained and results added to the files...
